# AgenticML — Experimento 3: California Housing (Regressão)

**Dataset:** California Housing (sklearn)  
**Tipo de problema:** Regressão (valor mediano de imóveis)  
**Objetivo:** Estimar o valor mediano das casas com base nas características demográficas e geográficas do bairro  
**Referência de comparação:** Auto-sklearn (seção 9.1 da spec)  

---
**Relevância:** Valida o caminho de regressão do pipeline, incluindo:
- Agente 5: seleção por R² (critério de seleção), RMSE reportado adicionalmente
- Agente 5: candidatos Linear Regression, Random Forest, XGBoost, SVR
- Agente 3: StandardScaler em features com escalas muito distintas

> ⚠️ **Nota:** O dataset completo tem 20.640 linhas. Para reprodutibilidade mais rápida, 
> foi utilizada uma amostra estratificada de 1.000 linhas (`random_state=42`).

In [1]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing

from core.graph import build_graph
from core.state import initial_state
from tcc_eval import make_split, evaluate_agenticml

print('Ambiente configurado.')


Ambiente configurado.


## 1. Dataset — California Housing

| Atributo | Valor |
|----------|-------|
| Instâncias (original) | 20.640 |
| Instâncias (amostra) | 1.000 |
| Features | 8 (MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude) |
| Target | MedHouseVal (valor mediano em $100k) |
| Missing values | Nenhum |
| Fonte | `sklearn.datasets.fetch_california_housing()` |

In [2]:
housing = fetch_california_housing(as_frame=True)
df_full = housing.frame.copy()

# Amostra reprodutível de 1.000 linhas
df = df_full.sample(n=1000, random_state=42).reset_index(drop=True)

print(f'Shape (amostra): {df.shape}')
print(f'Shape (original): {df_full.shape}')
print(f'\nTarget (MedHouseVal) — estatísticas:')
print(df['MedHouseVal'].describe().round(4))
display(df.head())

Shape (amostra): (1000, 9)
Shape (original): (20640, 9)

Target (MedHouseVal) — estatísticas:
count    1000.0000
mean        2.0970
std         1.1643
min         0.3000
25%         1.2552
50%         1.8160
75%         2.7115
max         5.0000
Name: MedHouseVal, dtype: float64


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,1.6812,25.0,4.192201,1.022284,1392.0,3.877437,36.06,-119.01,0.47700
1,2.5313,30.0,5.039384,1.193493,1565.0,2.679795,35.14,-119.46,0.45800
2,3.4801,52.0,3.977155,1.185877,1310.0,1.360332,37.80,-122.44,5.00001
3,5.7376,17.0,6.163636,1.020202,1705.0,3.444444,34.28,-118.72,2.18600
4,3.7250,34.0,5.492991,1.028037,1063.0,2.483645,36.62,-121.93,2.78000


In [3]:
csv_bytes = df.to_csv(index=False).encode('utf-8')
print(f'CSV gerado: {len(csv_bytes):,} bytes')

CSV gerado: 91,700 bytes


## 2. Execução do Pipeline AgenticML

In [4]:
PROMPT = (
    "Estimar o valor mediano das casas (MedHouseVal, em centenas de milhares de dólares) "
    "com base nas características demográficas e geográficas do bloco censitário."
)

# Protocolo de avaliação (parecer da orientação): holdout isolado, estratificado por
# quantis da target (estabiliza a partição entre sementes diferentes).
SEED = 42
train_df, test_df, split_info = make_split(
    df, target='MedHouseVal', task='regression', test_size=0.2, seed=SEED
)
print(f"Split: {split_info['n_train']} treino / {split_info['n_test']} teste "
      f"(estratificado={split_info['stratified']}, seed={SEED})")

t0 = time.time()
ev = evaluate_agenticml(
    train_df, test_df, target='MedHouseVal', prompt=PROMPT,
    hint='regression', seed=SEED, return_state=True,
)
elapsed = time.time() - t0
result = ev['_final_state']

print(f'\n✓ Pipeline concluído em {elapsed:.2f}s')
print('Status final:', result.get('status'))


Split: 800 treino / 200 teste (estratificado=True, seed=42)

✓ Pipeline concluído em 24.51s
Status final: done


## 3. Resultados por Agente

### Agente 1 — Ingestão e Inspeção

In [5]:
inspection = result.get('inspection') or {}
schema = inspection.get('schema', {})

print('Shape detectado:', schema.get('shape'))
print('Encoding:', inspection.get('encoding_detected'))
print('Colunas:', schema.get('columns'))

stats = inspection.get('descriptive_stats', {})
display(pd.DataFrame(stats).T.round(4))

Shape detectado: [800, 9]
Encoding: utf-8
Colunas: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedHouseVal']


,nulls,mean,std,min,max,median
MedInc,0.0,3.9624,2.0571,0.6831,15.0001,3.4960
HouseAge,0.0,28.9050,12.1998,2.0000,52.0000,30.0000
AveRooms,0.0,5.3991,1.7817,2.0967,36.0755,5.1950
AveBedrms,0.0,1.0757,0.2226,0.5000,5.2736,1.0467
Population,0.0,1414.5500,1011.9853,8.0000,8652.0000,1189.5000
AveOccup,0.0,2.9461,0.7972,1.3130,9.2414,2.8329
Latitude,0.0,35.4794,2.0863,32.5700,41.7900,34.1950
Longitude,0.0,-119.4391,1.9521,-124.1400,-115.3800,-118.4050
MedHouseVal,0.0,2.1040,1.1720,0.3000,5.0000,1.8160


### Agente 2 — Diagnóstico de Qualidade

In [6]:
quality = result.get('quality') or {}
issues = quality.get('issues', [])

print(f"Overall severity: {quality.get('overall_severity')}")
print(f"Issues detectados: {len(issues)}\n")

if issues:
    for i in issues:
        print(f"[{i['severity'].upper()}] {i['type']}")
        print(f"  → {i['alert']}\n")
else:
    print('✓ Nenhum problema de qualidade detectado.')

Overall severity: none
Issues detectados: 0

✓ Nenhum problema de qualidade detectado.


### Agente 3 — Feature Engineering

In [7]:
features = result.get('features') or {}
transforms = features.get('transformations_applied', [])

print(f"Target:   {features.get('target_column')}")
print(f"Features: {features.get('feature_columns')}")
print()
for t in transforms:
    print(f"• {t['type']} em {t['columns']}")
    print(f"  → {t['justification']}")

Target:   MedHouseVal
Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

• standard_scaler em ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
  → A padroniza e7 e3o  e9 recomendada para colunas num e9ricas quando n e3o h e1 limites naturais claros e para garantir que todas as features contribuam igualmente para o modelo, especialmente em algoritmos sens edveis a outliers.


### Agente 4 — Interpretação de Linguagem Natural

In [8]:
interp = result.get('interpretation') or {}
print(f"Tipo:       {interp.get('problem_type')}")
print(f"Confiança:  {interp.get('confidence')}")
print(f"Divergiu:   {interp.get('diverged_from_user')}")
print(f"\nJustificativa: {interp.get('justification')}")

Tipo:       regression
Confiança:  high
Divergiu:   False

Justificativa: O objetivo do usuário é estimar o valor mediano das casas, que é uma variável numérica contínua representada pela coluna 'MedHouseVal'. As características do dataset incluem variáveis demográficas e geográficas, todas numéricas, que são usadas para prever esse valor. A tarefa de prever um valor numérico contínuo a partir de outras variáveis caracteriza um problema de regressão. Portanto, a sugestão do usuário de que se trata de um problema de regressão está correta.


### Agente 5 — Seleção e Treinamento

Para regressão: critério de seleção = **R²** (melhor que prever a média). RMSE também reportado.

In [9]:
training = result.get('training') or {}

print(f"Modelo selecionado: {training.get('model_selected')}")
print(f"R² (cross-val, seleção): {training.get('metric_value')}")
if 'rmse' in training:
    print(f"RMSE (treino completo):  {training.get('rmse')}")
print()

candidates = training.get('candidate_results', [])
df_cands = pd.DataFrame(candidates).sort_values('score', ascending=False)
display(df_cands)

print(f"\nJustificativa: {training.get('justification')}")

print('\n=== Validação honesta (holdout isolado, nunca visto na seleção) ===')
print(f"R² no holdout:    {ev['holdout_metric']}")
print(f"RMSE no holdout:  {ev['holdout_rmse']}")
print(f"Gap de otimismo (CV − holdout): {ev['optimism_gap']}")
if ev['optimism_gap'] is not None and abs(ev['optimism_gap']) > 0.02:
    print('  ⚠ Gap > 0.02 — reportar o valor de holdout no TCC, não o de CV.')


Modelo selecionado: XGBoost
R² (cross-val, seleção): 0.741
RMSE (treino completo):  0.1508



,model,score
2,XGBoost,0.7410
3,SVR,0.7191
1,Random Forest,0.7167
0,Linear Regression,0.6647



Justificativa: XGBoost foi selecionado com R² de 0.7410 para o problema de regression. Demais candidatos avaliados: SVR (0.7191), Random Forest (0.7167), Linear Regression (0.6647). Hiperparâmetros otimizados via busca aleatória (CV): {'learning_rate': 0.0766, 'max_depth': 4, 'n_estimators': 314, 'subsample': 0.6232}.

=== Validação honesta (holdout isolado, nunca visto na seleção) ===
R² no holdout:    0.745
RMSE no holdout:  0.572
Gap de otimismo (CV − holdout): -0.004


### Agente 6 — Previsão e Explicação

In [10]:
output = result.get('output') or {}
explanation = output.get('explanation', {})
importances = output.get('feature_importances', [])

print('=== Explicação Técnica ===')
print(explanation.get('technical', ''))

print('\n=== Explicação Acessível ===')
print(explanation.get('accessible', ''))

if importances:
    print('\n=== Importância das Features ===')
    display(pd.DataFrame(importances))

=== Explicação Técnica ===
O pipeline AgenticML foi executado com sucesso, começando com a verificação de qualidade dos dados pelo Agente 2, que não encontrou problemas. O Agente 3 aplicou transformações de normalização usando o `standard_scaler` nas variáveis preditoras, como 'MedInc', 'HouseAge', entre outras, para garantir que todas as features estivessem na mesma escala, o que é crucial para o desempenho de muitos algoritmos de aprendizado de máquina. O Agente 4 confirmou que o problema é de regressão, pois o objetivo é prever o valor mediano das casas ('MedHouseVal'), uma variável contínua.

No estágio de treinamento, o Agente 5 selecionou o modelo XGBoost, que apresentou o melhor desempenho com um R² de 0.741, indicando que o modelo explica 74,1% da variância nos dados. O RMSE foi de 0.1508, sugerindo um erro médio razoável nas previsões. O XGBoost superou outros modelos como Regressão Linear (R²=0.6647), Random Forest (R²=0.7167) e SVR (R²=0.7191). A escolha do XGBoost foi justi

,feature,importance
0,MedInc,0.393015
1,AveOccup,0.163710
2,Longitude,0.102073
3,Latitude,0.094557
4,HouseAge,0.080409
5,AveRooms,0.055873
6,AveBedrms,0.055406
7,Population,0.054956


## 4. Resumo

In [11]:
schema = (result.get('inspection') or {}).get('schema', {})
issues = (result.get('quality') or {}).get('issues', [])
transforms = (result.get('features') or {}).get('transformations_applied', [])

summary = {
    'Experimento':              'California Housing — Regressão',
    'Dataset':                  f"{schema.get('shape', [0,0])[0]} linhas × {schema.get('shape', [0,0])[1]} colunas",
    'Split':                    f"{split_info['n_train']} treino / {split_info['n_test']} teste (seed={SEED})",
    'Tipo de problema':         interp.get('problem_type', 'N/A'),
    'Issues de qualidade':      len(issues),
    'Transformações':           len(transforms),
    'Modelo selecionado':       training.get('model_selected', 'N/A'),
    'Métrica (seleção)':        'R²',
    'AgenticML R² (CV)':        training.get('metric_value', 'N/A'),
    'AgenticML R² (holdout)':   ev['holdout_metric'],
    'AgenticML RMSE (holdout)': ev['holdout_rmse'],
    'Gap de otimismo':          ev['optimism_gap'],
    'Tempo (s)':                round(elapsed, 2),
}

display(pd.DataFrame([summary]).T.rename(columns={0: 'Valor'}))


,Valor
Experimento,California Housing — Regressão
Dataset,800 linhas × 9 colunas
Split,800 treino / 200 teste (seed=42)
Tipo de problema,regression
Issues de qualidade,0
Transformações,1
Modelo selecionado,XGBoost
Métrica (seleção),R²
AgenticML R² (CV),0.741
AgenticML R² (holdout),0.745
